# Get corekit from GitHub

coregit is a library of useful utilities that we can utilize for generation and tuning

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
if 'jrcai_corekit' not in os.listdir('.'):
    !git clone https://${GT_TOKEN}@github.com/MagedSaeed/jrcai_corekit.git
else: # else, pull latest changes
    !cd jrcai_corekit && git pull && cd ..

In [ ]:
!pip install -r jrcai_corekit/requirements.txt

add jrcai_corekit to path

In [ ]:
import sys
sys.path.append('jrcai_corekit/src')

check everything is working

In [ ]:
from llm.text_generator import TextGenerator

# Constants

In [ ]:
TAWJEEH_DATASET_NAME = 'ArEntail'
HF_EXPERIMENTAL_DATASET_NAME = 'MagedSaeed/ArEntail_experimental'
MODEL_PATH = "/raid_storage/shared_models/Qwen3-8B-Base"
MODEL_NAME = "Qwen3-8B"
TASK_NAME='NLI'

In [ ]:
TOKENIZER_PATH = MODEL_PATH

# Building the prompts dataset

In [ ]:
import requests

from tqdm.auto import tqdm

prompts = None

tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://promptlab.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts: raise Exception('Failed to fetch prompts')
prompts[:5]

In [ ]:
len(prompts)

In [ ]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED' and prompt['text_direction'].lower() == 'ltr', prompts))
len(filtered_prompts)

## Finetuning

### Get the dataset prompts

In [ ]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: TAWJEEH_DATASET_NAME in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(dataset_prompts)

In [ ]:
SELECTED_PROMPTS_IDS = [
    14581,
    14816,
    14818,
    14819,
    14820,
]

In [ ]:
dataset_prompts = list(filter(lambda prompt: prompt['id'] in SELECTED_PROMPTS_IDS, filtered_prompts))
len(dataset_prompts)

### Download the dataset

In [ ]:
import datasets

In [ ]:
hf_exp_dataset = datasets.load_dataset(HF_EXPERIMENTAL_DATASET_NAME)
hf_exp_dataset

### Merge the prompts

In [ ]:
from jinja2 import Environment, StrictUndefined

In [ ]:
import re
def preprocess_template(template):
    # remove punc at the end
    prefix,suffix = template.split('|||')
    # remove multi spaces
    # prefix = re.sub(r'\s+', ' ', prefix)
    return f'{prefix.strip()}\n{suffix.strip()}' # output is always the last line!

In [ ]:
def apply_template(prompt_template, sample):
    try:
        template = prompt_template['template']
        template = preprocess_template(template)
        sample['answer_choices'] = prompt_template['answer_choices']
        env = Environment(undefined=StrictUndefined)
        template = env.from_string(template)
        rendered_template = template.render(**sample)
        return rendered_template
    except Exception as e:
        print(prompt_template)
        print(sample)
        raise e

see how the template is applied on different examples

### Perform prompt-merge on one example prompt, for experimentation

In [ ]:
example_prompt_template = dataset_prompts[0]
print(apply_template(example_prompt_template, hf_exp_dataset['train'][3]))

In [ ]:
step_size = len(hf_exp_dataset['train'])/len(dataset_prompts)
step_size

In [ ]:
rendered_train_prompts_dataset = list()
for i,sample in enumerate(tqdm(hf_exp_dataset['train'])):
    if i % step_size == 0:
        print(f'rending {dataset_prompts[int(i/step_size)]["template"]}','sample index:',i)
    rendered_train_prompts_dataset.append(
        apply_template(dataset_prompts[int(i/step_size)], sample)
    )
len(rendered_train_prompts_dataset)

## Finetune the LLM

In [ ]:
GLOBAL_SEED = 42

In [ ]:
import random
random.seed(GLOBAL_SEED)

In [ ]:
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from llm import train_llm, LLMLoader, Qwen3Initializer, LoRAConfigRepository
from sklearn.model_selection import train_test_split

In [ ]:
llm_loader = LLMLoader(
        MODEL_PATH,
        llm_initializer=Qwen3Initializer(),
)
llm_loader

In [ ]:
model, tokenizer, generation_config = llm_loader()

In [ ]:
import re

train_samples,eval_samples = train_test_split(
    rendered_train_prompts_dataset,
    test_size=0.1,
    random_state=GLOBAL_SEED,
)

def generate_tuple(sample):
    sample_lines = sample.splitlines()
    # each sample should be two lines only, the first line is for the inputs and the last is the output
    # we did the join for generalization
    prefix = '\n'.join(sample_lines[:-1])
    prefix = prefix.strip()
    prefix += '\nThe answer is:'
    # outputs are always the last line
    suffix = sample_lines[-1].strip()
    suffix = f' {suffix}' # adding this space is important to split between input and output
    return prefix,suffix

train_samples = list(map(generate_tuple,train_samples))
eval_samples = list(map(generate_tuple,eval_samples))
len(train_samples), len(eval_samples), train_samples[:5], eval_samples[:5]

In [ ]:
train_llm(
    model=model,
    tokenizer=tokenizer,
    train_samples=train_samples,
    eval_samples=eval_samples,
    peft_config=LoRAConfigRepository.llama_3(),
    learning_rate=2.5e-4,
    epochs_count=10,
    train_batch_size=16,
    eval_batch_size=16,
    output_dir=f'Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/tuned_models/{MODEL_NAME}'
)

In [ ]:
exit()